# Парсинг данных с somon.tj

In [14]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://somon.tj"
UA = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123 Safari/537.36",
]

def get_html(url: str, timeout=25):
    r = requests.get(
        url,
        headers={"User-Agent": random.choice(UA), "Accept-Language": "ru,en;q=0.8"},
        timeout=timeout,
    )
    r.raise_for_status()
    return r.text

def norm_url(href: str):
    if not href:
        return None
    if href.startswith("/"):
        return urljoin(BASE, href)
    if href.startswith("http") and "somon.tj" in href:
        return href
    return None

def uniq(seq):
    seen = set()
    out = []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def parse_list_links(html: str):
    soup = BeautifulSoup(html, "lxml")
    # минимально и по твоим якорям: mask + card__title-link
    links = []
    for a in soup.select('a.mask[href*="/adv/"], a.card__title-link[href*="/adv/"]'):
        u = norm_url(a.get("href"))
        if u and "/adv/" in u:
            links.append(u)
    return uniq(links)
    

LIST_URL = "https://somon.tj/nedvizhimost/prodazha-kvartir/"
PAGES = 1          # сколько страниц листинга
SLEEP = (0.3, 0.9) # чтобы не долбить сайт

def stext(x):
    return re.sub(r"\s+", " ", (x or "")).strip()

def pick_int(s):
    if not s: return None
    m = re.search(r"\d[\d\s\xa0]*", s)
    if not m: return None
    return int(re.sub(r"[\s\xa0]", "", m.group(0)))

def pick_float_m2(s):
    if not s: return None
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*(?:м2|м²|m2)\b", s, re.I)
    if not m: return None
    return float(m.group(1).replace(",", "."))

def meta_content(soup, *keys):
    # keys: ("property","product:price:amount") etc
    for attr, val in keys:
        tag = soup.find("meta", attrs={attr: val})
        if tag and tag.get("content"):
            return tag["content"].strip()
    return None

def extract_price(soup):
    # 1) пробуем meta price (часто самый стабильный путь)
    for s in [
        meta_content(soup, ("property", "product:price:amount")),
        meta_content(soup, ("property", "og:price:amount")),
        meta_content(soup, ("name", "price")),
        meta_content(soup, ("itemprop", "price")),
    ]:
        p = pick_int(s)
        if p:
            return p

    # 2) иначе берём “самое большое” число из блоков, где в class/id есть слово price
    cands = []
    for tag in soup.select('[class*="price"], [id*="price"]'):
        t = stext(tag.get_text(" ", strip=True))
        v = pick_int(t)
        if v:
            cands.append(v)
    return max(cands) if cands else None

def parse_ad(html: str, url: str):
    soup = BeautifulSoup(html, "lxml")

    h1 = soup.find("h1")
    title = stext(h1.get_text(" ", strip=True)) if h1 else None

    # KV характеристики (твой стабильный блок)
    kv = {}
    for li in soup.select("div.announcement-characteristics ul.chars-column li"):
        t = stext(li.get_text(" ", strip=True))
        if ":" in t:
            k, v = t.split(":", 1)
            kv[stext(k).lower()] = stext(v)

    price = extract_price(soup)
    area_m2 = pick_float_m2(kv.get("площадь") or title)
    rooms = None
    if title:
        m = re.search(r"(\d+)\s*-\s*комн", title)
        rooms = int(m.group(1)) if m else None
    floor = pick_int(kv.get("этаж"))

    images = []
    for img in soup.select("img[src], img[data-src]"):
        src = img.get("src") or img.get("data-src")
        if src and "cdntj.somon.tj" in src:
            images.append(src)
    images = uniq(images)[:20]

    return {
        "url": url,
        "title": title,
        "price": price,
        "rooms": rooms,
        "area_m2": area_m2,
        "floor": floor,
        "build_type": kv.get("тип застройки"),
        "renovation": kv.get("ремонт"),
        "bathroom": kv.get("санузел"),
        "district": kv.get("район"),
        "heating": kv.get("отопление"),
        "condition": kv.get("состояние"),
        "techpassport": kv.get("техпаспорт"),
        "images": "|".join(images) if images else None,
        "kv_raw": str(kv) if kv else None,
    }

# ---- RUN ----
all_links = []
for p in range(1, PAGES + 1):
    sep = "&" if "?" in LIST_URL else "?"
    url = f"{LIST_URL}{sep}page={p}"
    html = get_html(url)
    all_links += parse_list_links(html)
    time.sleep(random.uniform(*SLEEP))

links = uniq(all_links)
print("Collected links:", len(links))

rows = []
for u in links:
    try:
        html = get_html(u)
        rows.append(parse_ad(html, u))
    except Exception as e:
        rows.append({"url": u, "error": str(e)})
    time.sleep(random.uniform(*SLEEP))

df = pd.DataFrame(rows)
df.head(3)


Collected links: 94


,url,title,price,rooms,area_m2,floor,build_type,renovation,bathroom,district,heating,condition,techpassport,images,kv_raw
0,https://somon.tj/adv/14823484_2-komn-kvartira-...,"2-комн. квартира, 3 этаж, 94 м², 32 мкр",449500.0,2.0,94.0,3.0,Новостройка,Без ремонта (коробка),Раздельный,32 мкр,Нет,Построено,Нет,https://cdntj.somon.tj/media/cache1/e0/b9/e0b9...,"{'площадь': '94 м²', 'тип застройки': 'Новостр..."
1,https://somon.tj/adv/13230613_2-komn-kvartira-...,"2-комн. квартира, 15 этаж, 128 м², Ашан",689000.0,2.0,128.0,15.0,Новостройка,Без ремонта (коробка),Раздельный,Ашан,Есть,На стадии строительства,Нет,https://cdntj.somon.tj/media/cache1/df/40/df40...,"{'площадь': '128 м²', 'тип застройки': 'Новост..."
2,https://somon.tj/adv/14925688_1-komn-kvartira-...,"1-комн. квартира, 5 этаж, 50 м², Чор-дома, Шох...",389000.0,1.0,50.0,5.0,Новостройка,Без ремонта (коробка),Раздельный,"Чор-дома, Шохмансур",Есть,На стадии строительства,Нет,https://cdntj.somon.tj/media/cache1/37/54/3754...,"{'площадь': '50 м²', 'тип застройки': 'Новостр..."


# Импорт библиотек

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from catboost import CatBoostRegressor


# Импорт данных

In [95]:
df = pd.read_csv("somon_ml.csv").drop(columns=["url"])
df

,price,rooms,area_m2,floor,district,build_type,renovation,bathroom,heating,condition,techpassport
0,899000,3,91.0,12.0,Сино 102 мкр,Новостройка,Новый ремонт,Раздельный,Есть,Построено,Есть
1,1023000,2,58.0,6.0,Бустон Сити (Гуля Голд),Новостройка,Новый ремонт,Совмещенный,Есть,Построено,Есть
2,565000,3,78.0,9.0,Шоҳмансур,Новостройка,Без ремонта (коробка),Раздельный,Есть,На стадии строительства,Нет
3,820000,2,65.0,5.0,20 мкр,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Нет
4,440000,2,60.0,6.0,Ленинский район,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Есть
...,...,...,...,...,...,...,...,...,...,...,...
11651,510000,1,42.0,1.0,82 мкр,Вторичный рынок,С ремонтом,Совмещенный,Нет,Построено,Есть
11652,890000,3,90.0,13.0,И Сомони Овир,Новостройка,Без ремонта (коробка),Раздельный,Есть,Построено,Нет
11653,1146000,2,70.0,6.0,славянский университет Себистон,Новостройка,Новый ремонт,Совмещенный,Есть,Построено,Нет
11654,680000,2,48.0,10.0,дом печати,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Нет


# Обработка данных

In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11656 entries, 0 to 11655
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         11656 non-null  int64  
 1   rooms         11656 non-null  int64  
 2   area_m2       11656 non-null  float64
 3   floor         11516 non-null  float64
 4   district      11656 non-null  object 
 5   build_type    11656 non-null  object 
 6   renovation    11656 non-null  object 
 7   bathroom      11652 non-null  object 
 8   heating       11652 non-null  object 
 9   condition     11656 non-null  object 
 10  techpassport  11535 non-null  object 
dtypes: float64(2), int64(2), object(7)
memory usage: 1001.8+ KB


In [97]:
columns = df.columns
print(f"🔵 df shape: {df.shape}")
print(f"🔵 df columns: {columns}")
print("--------------------------------\n\n")

for col in columns:
    print(f"🔵 {col}")
    print(f"dtype: {df[col].dtype}")
    print(f"unique count: {df[col].nunique()}")
    print(f"unique values: {df[col].unique()}")
    print(f"null count: {df[col].isnull().sum()}")
    



🔵 df shape: (11656, 11)
🔵 df columns: Index(['price', 'rooms', 'area_m2', 'floor', 'district', 'build_type',
       'renovation', 'bathroom', 'heating', 'condition', 'techpassport'],
      dtype='object')
--------------------------------


🔵 price
dtype: int64
unique count: 1879
unique values: [ 899000 1023000  565000 ...  132000 2430000 2230000]
null count: 0
🔵 rooms
dtype: int64
unique count: 6
unique values: [3 2 4 1 5 6]
null count: 0
🔵 area_m2
dtype: float64
unique count: 230
unique values: [9.100e+01 5.800e+01 7.800e+01 6.500e+01 6.000e+01 1.000e+02 9.000e+01
 1.010e+02 6.100e+01 7.500e+01 7.400e+01 8.600e+01 6.700e+01 9.400e+01
 1.230e+02 9.600e+01 8.200e+01 6.200e+01 5.000e+01 1.280e+02 9.500e+01
 1.260e+02 5.300e+01 3.700e+01 4.800e+01 9.800e+01 6.400e+01 5.400e+01
 4.400e+01 4.100e+01 4.900e+01 4.300e+01 4.500e+01 4.000e+01 7.700e+01
 1.100e+02 8.000e+01 1.360e+02 1.350e+02 7.300e+01 8.500e+01 1.150e+02
 5.600e+01 9.700e+01 6.800e+01 8.800e+01 1.120e+02 7.600e+01 8.300e+01
 1

In [98]:
df1 = df.dropna()
df1["area_m2"] = df1["area_m2"].astype(int)
df1["floor"] = df1["floor"].astype(int)
df1["rooms"] = df1["rooms"].astype(int)
df1["price"] = df1["price"].astype(int)

print(f"🔵 shape: {df1.shape}")
print(f"🔵 columns: {columns}")
print("--------------------------------\n\n")

for col in columns:
    print(f"🔵 {col}")
    print(f"dtype: {df1[col].dtype}")
    print(f"unique count: {df1[col].nunique()}")
    print(f"unique values: {df1[col].unique()}")
    print(f"null count: {df1[col].isnull().sum()}")

🔵 shape: (11398, 11)
🔵 columns: Index(['price', 'rooms', 'area_m2', 'floor', 'district', 'build_type',
       'renovation', 'bathroom', 'heating', 'condition', 'techpassport'],
      dtype='object')
--------------------------------


🔵 price
dtype: int64
unique count: 1834
unique values: [ 899000 1023000  565000 ...  132000 2430000 2230000]
null count: 0
🔵 rooms
dtype: int64
unique count: 6
unique values: [3 2 4 1 5 6]
null count: 0
🔵 area_m2
dtype: int64
unique count: 227
unique values: [  91   58   78   65   60  100   90  101   61   75   74   86   67   94
  123   96   82   62   50  128   95  126   53   37   48   98   64   54
   44   41   49   45   77  110   80  136  135   73   85  115   56   97
   68   88  112   76   83   43  122   70   47   87  108  130  142   72
   46   30   59   81   42  119  120   71   57   40   55   28   66   51
   92  102   63  144   26  134   93   52   89  114  140   84  107   69
  132  156  147  104  159  160  141   27  105  103  116  117  309   36
  118  165

C:\Users\oamir\AppData\Local\Temp\ipykernel_26988\3454414266.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["area_m2"] = df1["area_m2"].astype(int)
C:\Users\oamir\AppData\Local\Temp\ipykernel_26988\3454414266.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["floor"] = df1["floor"].astype(int)
C:\Users\oamir\AppData\Local\Temp\ipykernel_26988\3454414266.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

In [99]:
import re

DISTRICT_KEYWORDS = [

    # ===================== СИНО =====================
    ("Сино", [
        r'\bсино\b',
        r'\bн\.?\s*сино\b',

        # микрорайоны
        r'\b(8|12|13|14|17|18|19|20|29|30|31|32|33|34|82|83|84|91|92|101|102|103|104|112)\s*мкр\b',

        # участки и ориентиры
        r'\bказокон\b',
        r'\bкараболо\b',
        r'\bиспечак(-?\d)?\b',
        r'\bпайкар\b',
        r'\bзарафшон\b',
        r'\bзарнисор\b',
        r'\bчал-?чам\b',
        r'\bпрофсоюз\b',
        r'\bэстакада\b',
        r'\bавтовокзал\b',
        r'\bгипрозем\b',
        r'\bбарки\s*точик\b',
    ]),

    # ===================== ФИРДАВСИ =====================
    ("Фирдавси", [
        r'\bфирдавси\b',
        r'\bфирдавсӣ\b',
        r'\bн\.?\s*фирдавси\b',

        # микрорайоны
        r'\b(46|61|62|63|64|65)\s*мкр\b',

        # участки
        r'\bмардон\b',
        r'\bмолод[её]жный\b',
        r'\bсултони\s*кабир\b',
        r'\bюжный\b',
        r'\bсаховат\b',
        r'\bкорвон\b',
        r'\bмясокомбинат\b',
    ]),

    # ===================== ШОХМАНСУР =====================
    ("Шохмансур", [
        r'\bшохмансур\b',
        r'\bшоҳмансур\b',
        r'\bн\.?\s*шохмансур\b',

        # участки
        r'\bсадбарг\b',
        r'\bчулочк[аи]\b',
        r'\bзел[её]ный\s*базар\b',
        r'\bдом\s*печати\b',
        r'\bопера\b',
        r'\bбалет\b',
        r'\bхилтон\b',
        r'\bватан\b',
        r'\bбустон\s*сити\b',
        r'\bголубой\s*экран\b',
        r'\bстарый\s*аэропорт\b',
        r'\bаэропорт\b',
        r'\bпулоди\b',
        r'\bбофанда\b',
    ]),

    # ===================== И. СОМOНИ =====================
    ("И. Сомони", [
        r'\bисмоили\s*сомони\b',
        r'\bисмоил\s*сомони\b',
        r'\bи\.?\s*сомони\b',

        # ключевые
        r'\bовир\b',
        r'\bспартак\b',
        r'\bцум\b',
        r'\bрудаки\b',
        r'\bашан\b',
        r'\bславянск(ий|ого)\b',
        r'\bпарламент\b',
        r'\bнацбанк\b',
        r'\bальфемо\b',
        r'\bботанический\s*сад\b',
        r'\bпарк\s*айни\b',
        r'\bводонасос\b',
    ]),
]

def assign_district(text: str) -> str:
    if not isinstance(text, str):
        return "Другое"

    t = text.lower()

    for district, patterns in DISTRICT_KEYWORDS:
        for p in patterns:
            if re.search(p, t):
                return district

    return "Другое"


df1["district"] = df1["district"].apply(assign_district)

print(list(df1["district"].unique()))
print(df1["district"].nunique())

['Сино', 'Шохмансур', 'Другое', 'Фирдавси', 'И. Сомони']
5


C:\Users\oamir\AppData\Local\Temp\ipykernel_26988\2078611614.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["district"] = df1["district"].apply(assign_district)


In [103]:
df1["district"].value_counts()

district
Сино         3943
Другое       3208
Шохмансур    1743
И. Сомони    1564
Фирдавси      916
Name: count, dtype: int64

In [ ]:
# df1.to_csv("somon_ml_clear_amir2.csv", index=False)

In [114]:
df1 = pd.read_csv("somon_ml_clear_amir2.csv")
df1

,price,rooms,area_m2,floor,district,build_type,renovation,bathroom,heating,condition,techpassport
0,899000,3,91,12,Сино,Новостройка,Новый ремонт,Раздельный,Есть,Построено,Есть
1,1023000,2,58,6,Шохмансур,Новостройка,Новый ремонт,Совмещенный,Есть,Построено,Есть
2,565000,3,78,9,Шохмансур,Новостройка,Без ремонта (коробка),Раздельный,Есть,На стадии строительства,Нет
3,820000,2,65,5,Сино,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Нет
4,440000,2,60,6,Другое,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Есть
...,...,...,...,...,...,...,...,...,...,...,...
11369,510000,1,42,1,Сино,Вторичный рынок,С ремонтом,Совмещенный,Нет,Построено,Есть
11370,890000,3,90,13,И. Сомони,Новостройка,Без ремонта (коробка),Раздельный,Есть,Построено,Нет
11371,1146000,2,70,6,И. Сомони,Новостройка,Новый ремонт,Совмещенный,Есть,Построено,Нет
11372,680000,2,48,10,Шохмансур,Новостройка,Новый ремонт,Совмещенный,Нет,Построено,Нет


In [108]:
df1 = pd.read_csv("somon_ml_clear_amir2.csv")

print(f"🔵 before {df1.shape}")
df1.drop_duplicates(inplace=True)

columns_except_price = [col for col in df1.columns if col != "Цена"]
df1_no_duplicates = df1.drop_duplicates(subset=columns_except_price)
df1 = df1[(df1["price"] > 100000) & (df1["price"] < 10000000)]
print(f"🔵 after {df1.shape}")


🔵 before (11374, 11)
🔵 after (10070, 11)


# Train Test Split

In [109]:
X = df1.drop(columns=["price"])
y = df1["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle

df1_shuffled = shuffle(df1, random_state=42).reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ohe = OneHotEncoder(handle_unknown="ignore")
X_train_ohe = ohe.fit_transform(X_train[["district"]])
X_test_ohe = ohe.transform(X_test[["district"]])



joblib.dump(ohe, 'ohe_encoder.joblib')

# Загрузка
# ohe_loaded = joblib.load('ohe_encoder.joblib')

In [93]:
from sklearn.ensemble import RandomForestRegressor
model1 = RandomForestRegressor(
    n_estimators=1000,
    random_state=42,
)
model1.fit(X_train_ohe, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsampl

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model1.predict(X_test_ohe)


import numpy as np
mae = mean_absolute_error(y_test, y_pred)
mse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"R2: {r2}")  


